In [9]:
from itertools import combinations
import numpy as np

## Simple Test Problem

In [10]:
CITIES = [
    "Rome",
    "Milan",
    "Naples",
    "Turin",
    "Palermo",
    "Genoa",
    "Bologna",
    "Florence",
    "Bari",
    "Catania",
    "Venice",
    "Verona",
    "Messina",
    "Padua",
    "Trieste",
    "Taranto",
    "Brescia",
    "Prato",
    "Parma",
    "Modena",
]
test_problem = np.load('LAB2-data/test_problem.npy')
test_problem.shape

(20, 20)

## Common tests

In [11]:
problem = np.load('LAB2-data/problem_r2_100.npy')

In [12]:
# Negative values?
np.any(problem < 0)

np.True_

In [13]:
# Diagonal is all zero?
np.allclose(np.diag(problem), 0.0)

False

In [14]:
# Symmetric matrix?
np.allclose(problem, problem.T)

False

### Triangular inequality 
Perché è importante per il lab
Questo test nel notebook tsp.ipynb serve a farti capire la differenza tra i tipi di problema:

I problemi geometrici (g_) rispetteranno questa regola (il risultato sarà True).

I problemi casuali (r1_, r2_) molto probabilmente non la rispetteranno (il risultato sarà False, come puoi vedere dall'output nel notebook per problem_r2_100.npy).

In [15]:
# Triangular inequality
all(
    problem[x, y] <= problem[x, z] + problem[z, y]
    for x, y, z in list(combinations(range(problem.shape[0]), 3))
)

False

In [23]:
import numpy as np
import glob
from itertools import combinations

print("--- TEST DI SIMMETRIA (O(N^2) - Veloce) ---")
print("Controlla se dist(A, B) == dist(B, A)")

# Prendo tutti i file
problem_files = glob.glob("LAB2-data/*.npy")
problem_files = [f for f in problem_files if 'test_problem' not in f]
problem_files.sort()

def get_name(path):
    return path.replace('\\', '/').split('/')[-1]

for problem_file_path in problem_files:
    problem_name = get_name(problem_file_path)
    problem = np.load(problem_file_path)
    
    # np.allclose è il modo corretto per confrontare float
    is_symmetric = np.allclose(problem, problem.T)
    
    print(f"File: {problem_name:<20} | È Simmetrico? -> {is_symmetric}")

print("\nTest di simmetria completato.")

--- TEST DI SIMMETRIA (O(N^2) - Veloce) ---
Controlla se dist(A, B) == dist(B, A)
File: problem_g_10.npy     | È Simmetrico? -> True
File: problem_g_100.npy    | È Simmetrico? -> True
File: problem_g_1000.npy   | È Simmetrico? -> True
File: problem_g_20.npy     | È Simmetrico? -> True
File: problem_g_200.npy    | È Simmetrico? -> True
File: problem_g_50.npy     | È Simmetrico? -> True
File: problem_g_500.npy    | È Simmetrico? -> True
File: problem_r1_10.npy    | È Simmetrico? -> False
File: problem_r1_100.npy   | È Simmetrico? -> False
File: problem_r1_1000.npy  | È Simmetrico? -> False
File: problem_r1_20.npy    | È Simmetrico? -> False
File: problem_r1_200.npy   | È Simmetrico? -> False
File: problem_r1_50.npy    | È Simmetrico? -> False
File: problem_r1_500.npy   | È Simmetrico? -> False
File: problem_r2_10.npy    | È Simmetrico? -> False
File: problem_r2_100.npy   | È Simmetrico? -> False
File: problem_r2_1000.npy  | È Simmetrico? -> False
File: problem_r2_20.npy    | È Simmetrico

## HILL CLIMBING

**Obbiettivo:** Trovare il percorso con il "punteggio" (costo) più **basso** possibile.

- **_100** (il numero): Indica la dimensione del problema, cioè il numero di città. In questo caso, 100 città. Troverai file con 10, 20, 50, 100, 200, 500, e 1000 città.

- **g_** (la lettera): Indica la natura della matrice delle distanze.

- **g (Geometric)**: Il problema è geometrico e simmetrico. Significa che la distanza da A a B è uguale alla distanza da B ad A. Queste matrici rispettano anche la disuguaglianza triangolare (andare da A a C non è mai più lungo che passare da A a B e poi a C).

- **r1 e r2 (Random)**: I problemi sono asimmetrici. Significa che la distanza da A a B è diversa dalla distanza da B ad A (pensa a un volo aereo o a strade a senso unico). Come puoi vedere nel notebook tsp.ipynb di partenza, il test np.allclose(problem, problem.T) fallisce per problem_r2_100.npy, confermando che la matrice non è simmetrica. r1 e r2 sono semplicemente due generatori diversi di problemi asimmetrici.



In breve:

g: TSP Simmetrico (facile)

r1, r2: TSP Asimmetrico (difficile)

Numero: Quante città.

In [24]:
import numpy as np
import random
import time
import glob
import pandas as pd
import math
from IPython.display import display




# --- Funzioni di Supporto---

def evaluate_solution(solution, distance_matrix):
    """
    Calcola il costo totale (distanza) di un percorso (soluzione).
    Operazione O(N).
    """
    total_cost = 0
    num_cities = len(solution)
    for i in range(num_cities):
        city_a = solution[i]
        city_b = solution[(i + 1) % num_cities] # % gestisce il ritorno all'inizio
        total_cost += distance_matrix[city_a, city_b]
    return total_cost

def get_random_solution(num_cities):
    """
    Genera una soluzione casuale (un percorso come lista di indici).
    """
    solution = list(range(num_cities))
    random.shuffle(solution)
    return solution



### Divisione dei problemi 

In [25]:

# Cerca tutti i file
problem_files = glob.glob("LAB2-data/*.npy")
problem_files = [f for f in problem_files if 'test_problem' not in f]

def get_name(path):
    return path.replace('\\', '/').split('/')[-1]

def get_sort_key(file_path):
    name = get_name(file_path)
    number_str = name.split('_')[-1] 
    number_str = number_str.split('.')[0]
    return int(number_str)


# 3 liste
g_files = [f for f in problem_files if get_name(f).startswith('problem_g')]
r1_files = [f for f in problem_files if get_name(f).startswith('problem_r1')]
r2_files = [f for f in problem_files if get_name(f).startswith('problem_r2')]

g_files = sorted(g_files, key=get_sort_key)
r1_files = sorted(r1_files, key=get_sort_key)
r2_files = sorted(r2_files, key=get_sort_key)

print(f"File G (Simmetrici): {len(g_files)}")
print(f"File R1 (Asimmetrici): {len(r1_files)}")
print(f"File R2 (Asimmetrici): {len(r2_files)}")



File G (Simmetrici): 7
File R1 (Asimmetrici): 7
File R2 (Asimmetrici): 7


In [ ]:
# --- Componenti dell'Algoritmo Genetico (GA) ---

def tournament_selection(population, fitnesses, k=3):
    # Seleziona k indici casuali (con rimpiazzo)
    selected_indices = [random.randrange(len(population)) for _ in range(k)]
    
    # Trova l'indice del migliore (fitness minore) tra i k selezionati
    best_index_in_tournament = min(selected_indices, key=lambda i: fitnesses[i])
    
    return population[best_index_in_tournament]



# --- Componenti dell'Algoritmo Genetico (GA) per TSP ---

def order_crossover(parent1, parent2):

    """ Esegue l'Order Crossover (OX1), un operatore standard per permutazioni """

    num_cities = len(parent1)
    start, end = sorted(random.sample(range(num_cities), 2))
    
    child = [None] * num_cities
    child[start:end+1] = parent1[start:end+1]
    cities_in_child = set(child)
    
    child_idx = (end + 1) % num_cities
    parent2_idx = (end + 1) % num_cities
    
    while None in child:
        city = parent2[parent2_idx]
        if city not in cities_in_child:
            child[child_idx] = city
            child_idx = (child_idx + 1) % num_cities
        parent2_idx = (parent2_idx + 1) % num_cities
        
    return child

def swap_mutation(solution):

    """ Esegue una semplice mutazione scambiando due città a caso """

    i, k = random.sample(range(len(solution)), 2)
    solution[i], solution[k] = solution[k], solution[i]
    return solution

In [27]:
def apply_local_search_2opt_fast(solution, distance_matrix):
    """
    Applica una ricerca locale 2-opt (Hill Climbing).
    Itera finché non trova un miglioramento.
    
    ATTENZIONE: Questa logica O(1) è VALIDA SOLO PER PROBLEMI SIMMETRICI (g_).
    """
    num_cities = len(solution)
    current_cost = evaluate_solution(solution, distance_matrix)
    
    improved = True
    while improved:
        improved = False
        
        # Iteriamo su tutte le possibili coppie (i, k)
        for i in range(num_cities - 1):
            for k in range(i + 1, num_cities):
                
                # Evita il caso che rompe lo stesso arco (wrap-around)
                if i == 0 and k == num_cities - 1:
                    continue

                city_i_minus_1 = solution[i - 1]
                city_i = solution[i]
                city_k = solution[k]
                city_k_plus_1 = solution[(k + 1) % num_cities]

                cost_removed = distance_matrix[city_i_minus_1, city_i] + distance_matrix[city_k, city_k_plus_1]
                cost_added = distance_matrix[city_i_minus_1, city_k] + distance_matrix[city_i, city_k_plus_1]
                
                delta_cost = cost_added - cost_removed
                # -----------------------------------------------------------

                if delta_cost < -1e-9: # Se c'è un miglioramento 
                    # mossa 2-opt (inversione)
                    solution = solution[:i] + solution[i:k+1][::-1] + solution[k+1:]
                    current_cost += delta_cost
                    improved = True
                    break # Esci dal ciclo interno e ricomincia
            if improved:
                break # Esci dal ciclo esterno e ricomincia
                
    return solution, current_cost



# --- Ricerca Locale O(1) per ASIMMETRICI (ATSP) ---

def apply_local_search_insert_ATSP(solution, distance_matrix):
    """
    Applica una ricerca locale (Hill Climbing) usando la mossa 'insert'.
    Questa mossa è corretta per ATSP e ha un delta O(1).
    Itera finché non trova un miglioramento.
    """
    num_cities = len(solution)
    current_cost = evaluate_solution(solution, distance_matrix) # Calcolo iniziale
    
    improved = True
    while improved:
        improved = False
        
        for k in range(num_cities): 
            # k è l'indice della città da SPOSTARE (la nostra 'c')
            
            # 1. Identifica 'c' e i suoi vicini 'y' e 'z'
            city_c = solution[k]
            city_y = solution[k - 1] # Gestito da indicizzazione negativa di Python
            city_z = solution[(k + 1) % num_cities]

            for i in range(num_cities):
                # i è l'indice della città DOPO CUI INSERIRE (la nostra 'a')
                
                if i == k or i == (k - 1 + num_cities) % num_cities:
                    # Salta se stiamo cercando di inserire 'c' accanto a se stessa
                    continue
                
                # 2. Identifica 'a' e il suo vicino 'b'
                city_a = solution[i]
                city_b = solution[(i + 1) % num_cities]

                # 3. Calcola il Delta O(1)
                cost_removed = distance_matrix[city_y, city_c] + distance_matrix[city_c, city_z] + distance_matrix[city_a, city_b]
                cost_added   = distance_matrix[city_y, city_z] + distance_matrix[city_a, city_c] + distance_matrix[city_c, city_b]
                
                delta_cost = cost_added - cost_removed

                if delta_cost < -1e-9: # Se c'è un miglioramento
                    
                    # 4. Applica la mossa (rimuovi 'c' e inseriscila dopo 'a')
                    # Questa è l'operazione O(N) che facciamo solo DOPO aver trovato un miglioramento
                    
                    # Converte in lista per manipolazione
                    sol_list = list(solution) 
                    
                    # Rimuovi 'c'
                    city_to_move = sol_list.pop(k)
                    
                    # Calcola il nuovo indice di 'a' (potrebbe essere cambiato se k < i)
                    new_i = sol_list.index(city_a)
                    
                    # Inserisci 'c' dopo 'a'
                    sol_list.insert(new_i + 1, city_to_move)

                    solution = sol_list # Aggiorna la soluzione
                    current_cost += delta_cost
                    improved = True
                    break # Esci dal ciclo 'i'
            if improved:
                break # Esci dal ciclo 'k' e ricomincia il 'while'
                
    return solution, current_cost

In [31]:
# --- MEMETIC ALGORITHM (GA + Local Search) ---

def memetic_algorithm_2opt(distance_matrix, 
                           population_size=100, 
                           generations=500, 
                           elite_size=10, 
                           mutation_rate=0.1,
                           local_search_rate=0.2):
    """
    Esegue un Algoritmo Memetico (GA + 2-opt Local Search).
    - elitismo: i 'elite_size' migliori sopravvivono sempre.
    - memetico: 'local_search_rate' % dei figli subisce una local search.
    
    ATTENZIONE: Usa questo algoritmo SOLO per i problemi SIMMETRICI (g_),
    poiché la sua local search (apply_local_search_2opt_fast) è tale.
    """
    num_cities = distance_matrix.shape[0]
    
    # --- 1. Inizializzazione ---
    population = []
    fitnesses = []
    for _ in range(population_size):
        solution = get_random_solution(num_cities)
        fitness = evaluate_solution(solution, distance_matrix)
        population.append(solution)
        fitnesses.append(fitness)
        
    # Teniamo traccia del migliore di sempre
    best_solution = min(population, key=lambda sol: evaluate_solution(sol, distance_matrix))
    best_cost = evaluate_solution(best_solution, distance_matrix)
    
    # print(f"Generazione 0: Miglior Costo = {best_cost:.2f}")

    # --- 2. Ciclo Evolutivo ---
    for gen in range(generations):
        new_population = []
        new_fitnesses = []
        
        # --- 3. Elitismo ---
        # Ordina la popolazione in base alla fitness (dalla migliore alla peggiore)
        sorted_indices = sorted(range(population_size), key=lambda k: fitnesses[k])
        
        for i in range(elite_size):
            elite_index = sorted_indices[i]
            new_population.append(population[elite_index])
            new_fitnesses.append(fitnesses[elite_index])
            
        # --- 4. Generazione Nuovi Figli ---
        while len(new_population) < population_size:
            # Selezione
            parent1 = tournament_selection(population, fitnesses)
            parent2 = tournament_selection(population, fitnesses)
            
            # Crossover
            child = order_crossover(parent1, parent2)
            
            # Mutazione
            if random.random() < mutation_rate:
                child = swap_mutation(child)
            
            # --- 5. Local Search (Parte Memetica) ---
            if random.random() < local_search_rate:
                child, child_cost = apply_local_search_2opt_fast(child, distance_matrix)
            else:
                child_cost = evaluate_solution(child, distance_matrix)

            new_population.append(child)
            new_fitnesses.append(child_cost)

        # Aggiorna la popolazione
        population = new_population
        fitnesses = new_fitnesses
        
        # Aggiorna il migliore di sempre
        current_best_index = min(range(population_size), key=lambda k: fitnesses[k])
        current_best_cost = fitnesses[current_best_index]
        
        if current_best_cost < best_cost:
            best_cost = current_best_cost
            best_solution = population[current_best_index]
            # print(f"Generazione {gen+1}: Nuovo Migliore = {best_cost:.2f}")
            
    return best_solution, best_cost




# --- Algoritmo 5: MEMETIC ALGORITHM per ATSP (r1_, r2_) ---

def memetic_algorithm_ATSP(distance_matrix, 
                           population_size=100, 
                           generations=500, 
                           elite_rate=0.2, 
                           mutation_rate=0.1,
                           local_search_rate=0.2,
                           tournament_k=5):
    """
    Esegue un Algoritmo Memetico (GA + 'insert' Local Search).
    
    *** SPECIFICO PER ATSP (r1_, r2_) ***
    Usa la logica di elitismo/selezione di symreg.ipynb
    Usa la local search 'apply_local_search_insert_ATSP' O(1) corretta.
    """
    num_cities = distance_matrix.shape[0]
    
    # --- 1. Inizializzazione ---
    population = []
    for _ in range(population_size):
        population.append(get_random_solution(num_cities))
        
    best_solution_ever = population[0]
    best_cost_ever = evaluate_solution(best_solution_ever, distance_matrix)

    # --- 2. Ciclo Evolutivo ---
    for gen in range(generations):
        
        # --- 3. Valutazione ---
        fitnesses = [evaluate_solution(sol, distance_matrix) for sol in population]
        
        # --- 4. Ordinamento e Logica Elitismo (da symreg.ipynb) ---
        sorted_indices = np.argsort(fitnesses)
        population = [population[i] for i in sorted_indices]
        fitnesses = [fitnesses[i] for i in sorted_indices]

        if fitnesses[0] < best_cost_ever:
            best_cost_ever = fitnesses[0]
            best_solution_ever = population[0]
            # print(f"Generazione {gen}: Nuovo Migliore = {best_cost_ever:.2f}")

        elite_size = int(population_size * elite_rate)
        new_population = [population[i].copy() for i in range(elite_size)]
        
        elite_pool = population[:elite_size]
        elite_fitnesses = fitnesses[:elite_size]

        # --- 5. Generazione Nuovi Figli ---
        while len(new_population) < population_size:
            
            # Selezione (da symreg.ipynb)
            parent1 = tournament_selection(elite_pool, elite_fitnesses, k=tournament_k)
            parent2 = tournament_selection(elite_pool, elite_fitnesses, k=tournament_k)
            
            # Crossover (specifico per TSP)
            child = order_crossover(parent1, parent2)
            
            # Mutazione (specifica per TSP)
            if random.random() < mutation_rate:
                child = swap_mutation(child) # Puoi anche provare 'insert' qui
            
            # --- 6. Local Search (Parte Memetica per ATSP) ---
            if random.random() < local_search_rate:
                # Applichiamo la NUOVA local search
                child, _ = apply_local_search_insert_ATSP(child, distance_matrix)

            new_population.append(child)

        population = new_population
            
    # Alla fine, rivaluta il migliore
    final_fitnesses = [evaluate_solution(sol, distance_matrix) for sol in population]
    best_index = np.argmin(final_fitnesses)
    final_best_solution = population[best_index]
    final_best_cost = final_fitnesses[best_index]

    if best_cost_ever < final_best_cost:
        return best_solution_ever, best_cost_ever
    else:
        return final_best_solution, final_best_cost

In [ ]:
# --- ESECUZIONE PROBLEMI GEOMETRICI (g_) con Algoritmo Memetico ---

print("\n--- ESECUZIONE PROBLEMI GEOMETRICI (g_) - ALGORITMO MEMETICO ---")
g_results = []

for problem_file_path in g_files:
    problem_name = get_name(problem_file_path)
    problem_matrix = np.load(problem_file_path)
    
    print(f"\n Processando (MA-2opt-FAST): {problem_name}")
    start_time = time.time()
    
    # REGOLA I PARAMETRI QUI
    # (pop_size, generations, elite_size, mutation_rate, local_search_rate)
    ma_sol, ma_cost = memetic_algorithm_2opt(
        problem_matrix,
        population_size=100, 
        generations=200, # Aumenta per problemi più grandi
        elite_size=10,
        mutation_rate=0.1,
        local_search_rate=0.25 # Applica local search al 25% dei figli
    )
    
    ma_time = time.time() - start_time
    
    print(f"    -> Risultato: Costo={ma_cost:.2f}, Tempo={ma_time:.4f}s")
    
    g_results.append({
        'Problem': problem_name,
        'MA Cost': ma_cost,
        'MA Time (s)': ma_time
    })

print("\n--- Risultati Categoria G (Memetico) ---")
g_df = pd.DataFrame(g_results)
display(g_df)






In [32]:

# --- ESECUZIONE PROBLEMI ASIMMETRICI (r1_) con Algoritmo Memetico (Asimmetrico) ---

print("\n--- ESECUZIONE PROBLEMI ASIMMETRICI (r1_) - MA (Asimmetrico) ---")
r1_results = []

for problem_file_path in r1_files:
    problem_name = get_name(problem_file_path)
    problem_matrix = np.load(problem_file_path)
    
    print(f"\n Processando (MA-Insert-FAST): {problem_name}")
    start_time = time.time()
    
    # REGOLA I PARAMETRI QUI
    ma_sol, ma_cost = memetic_algorithm_ATSP(
        problem_matrix,
        population_size=100, 
        generations=200,    # Aumenta per problemi più grandi
        elite_rate=0.2,     # 20% di élite
        mutation_rate=0.1,
        local_search_rate=0.25, # Applica 'insert' al 25% dei figli
        tournament_k=5
    )
    
    ma_time = time.time() - start_time
    
    print(f"    -> Risultato: Costo={ma_cost:.2f}, Tempo={ma_time:.4f}s")
    
    r1_results.append({
        'Problem': problem_name,
        'MA Cost': ma_cost,
        'MA Time (s)': ma_time
    })

print("\n--- Risultati Categoria R1 (Memetico) ---")
r1_df = pd.DataFrame(r1_results)
display(r1_df)


--- ESECUZIONE PROBLEMI ASIMMETRICI (r1_) - MA (Asimmetrico) ---

 Processando (MA-Insert-FAST): problem_r1_10.npy
    -> Risultato: Costo=184.27, Tempo=0.2959s

 Processando (MA-Insert-FAST): problem_r1_20.npy
    -> Risultato: Costo=340.06, Tempo=1.0382s

 Processando (MA-Insert-FAST): problem_r1_50.npy
    -> Risultato: Costo=544.05, Tempo=7.3367s

 Processando (MA-Insert-FAST): problem_r1_100.npy
    -> Risultato: Costo=728.99, Tempo=38.2298s

 Processando (MA-Insert-FAST): problem_r1_200.npy


KeyboardInterrupt: 